## EXPLORATION GOLD DATASET

In [130]:
#import pandas as pd
#import sqlalchemy as sa
#from indusense.db.session import create_postgres_engine
#from indusense.db.models import GoldMachineHourlyFeature

#engine = create_postgres_engine()
#stmt = sa.select(GoldMachineHourlyFeature)

#gold_df = pd.read_sql(stmt, engine)
#gold_df.head()
    

###############

import pandas as pd 

gold_df = pd.read_csv("C:\\Formation\\gold_dataset\\gold_dataset_20260526-104215.csv")
print (gold_df.head())

print (gold_df.isna().sum())
print(gold_df[gold_df.isna().any(axis=1)].head())

  machine_code         window_start           window_end  temp_mean_6h  \
0      MACH-04  2025-01-10 00:00:00  2025-01-10 01:00:00         37.51   
1      MACH-12  2025-01-10 10:00:00  2025-01-10 11:00:00         46.00   
2      MACH-02  2025-01-10 19:00:00  2025-01-10 20:00:00         42.34   
3      MACH-03  2025-01-12 05:00:00  2025-01-12 06:00:00         39.89   
4      MACH-05  2025-02-10 04:00:00  2025-02-10 05:00:00         33.11   

   temp_max_6h  temp_std_6h  pressure_mean_6h  pressure_max_6h  \
0        37.51          NaN               NaN              NaN   
1        46.00          NaN               NaN              NaN   
2        42.34          NaN               NaN              NaN   
3        39.89          NaN               NaN              NaN   
4        33.11          NaN               NaN              NaN   

   pressure_std_6h  temp_mean_12h  ...  type_arret_urgence_count_prev_24h  \
0              NaN          37.51  ...                                  0   
1   

In [131]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

In [132]:
# ===== SETUP MLFLOW =====
import mlflow
import mlflow.sklearn
import mlflow.xgboost

MLFLOW_URI = "file:C:/indusense/mlruns"
mlflow.set_tracking_uri(MLFLOW_URI)
mlflow.set_experiment("indusense-failure-prediction")

print(f"Tracking URI : {mlflow.get_tracking_uri()}")
print("Experiment   : indusense-failure-prediction")
print(f"\nPour lancer l'UI :")
print(f"  mlflow ui --backend-store-uri {MLFLOW_URI}")

Tracking URI : file:C:/indusense/mlruns
Experiment   : indusense-failure-prediction

Pour lancer l'UI :
  mlflow ui --backend-store-uri file:C:/indusense/mlruns


## Recommandation pour action sur les NaN
Si une colonne est critique et que les valeurs manquantes sont rares, dropna() est souvent acceptable.
Si les valeurs manquantes sont fréquentes, mieux vaut fillna() avec une moyenne/médiane ou une méthode d’interpolation.
Toujours commencer par inspecter.

En résumé : inspecter les NaN, puis choisir dropna, fillna, ou interpolate selon si vous pouvez supprimer les lignes ou si vous devez conserver et estimer les valeurs manquantes.

Supprimer les lignes contenant des NaN
gold_df_clean = gold_df.dropna()

Remplacer les NaN par une valeur fixe
gold_df_filled = gold_df.fillna(0)

Interpolation si les données sont temporelles / ordonnées
gold_df = gold_df.interpolate()

In [133]:
# Résumé concis des NaN
nan_summary = gold_df.isna().sum()
cols_with_nan = nan_summary[nan_summary > 0]

print(f"Dimensions: {gold_df.shape}")
print(f"\nColonnes avec NaN: {len(cols_with_nan)}")
if len(cols_with_nan) > 0:
    print(cols_with_nan)
    print(f"\nPourcentage de NaN (max): {(cols_with_nan.max() / len(gold_df) * 100):.2f}%")
else:
    print("✓ Aucun NaN dans le dataset!")
    
nan_pct = (gold_df.isna().sum() / len(gold_df) * 100)
nan_pct[nan_pct > 0].sort_values(ascending=False)

Dimensions: (66678, 45)

Colonnes avec NaN: 18
temp_std_6h                          15
pressure_mean_6h                   1117
pressure_max_6h                    1117
pressure_std_6h                    1146
temp_std_12h                         15
pressure_mean_12h                  1079
pressure_max_12h                   1079
pressure_std_12h                   1106
temp_std_24h                         15
pressure_mean_24h                  1007
pressure_max_24h                   1007
pressure_std_24h                   1034
temp_trend_6h                       300
pressure_trend_6h                  1504
temp_zscore_24h                     120
incident_max_severity_prev_24h    63465
hours_since_last_incident          8851
ambient_humidity_pct              66678
dtype: int64

Pourcentage de NaN (max): 100.00%


ambient_humidity_pct              100.000000
incident_max_severity_prev_24h     95.181319
hours_since_last_incident          13.274243
pressure_trend_6h                   2.255617
pressure_std_6h                     1.718708
pressure_mean_6h                    1.675215
pressure_max_6h                     1.675215
pressure_std_12h                    1.658718
pressure_max_12h                    1.618225
pressure_mean_12h                   1.618225
pressure_std_24h                    1.550736
pressure_mean_24h                   1.510243
pressure_max_24h                    1.510243
temp_trend_6h                       0.449924
temp_zscore_24h                     0.179969
temp_std_6h                         0.022496
temp_std_24h                        0.022496
temp_std_12h                        0.022496
dtype: float64

Stratégie recommandée par type de colonne
1. Supprimer complètement ambient_humidity_pct (100% de NaN)
2. Pour incident_max_severity_prev_24h (95% de NaN) → Supprimer la colonne ou remplacer par 0 (pas d'incident)
3. Pour hours_since_last_incident (13% de NaN) → Remplacer par la médiane ou -1 (pas d'incident connu)
4. Pour les colonnes pressure_* et temp_* (1-2% de NaN) → Interpolation (données temporelles)


In [134]:
# ===== MODULE DE NETTOYAGE DES NaN =====

print("=== NETTOYAGE DES NaN ===\n")

# 1. Supprimer les colonnes avec 100% de NaN
cols_to_drop = nan_pct[nan_pct == 100].index.tolist()
if cols_to_drop:
    print(f"1. Suppression de colonnes vides (100% NaN): {cols_to_drop}")
    gold_df = gold_df.drop(columns=cols_to_drop)
else:
    print("1. Aucune colonne avec 100% de NaN")

# 2. Traiter les colonnes avec > 50% de NaN
cols_high_nan = nan_pct[(nan_pct > 50) & (nan_pct < 100)].index.tolist()
if cols_high_nan:
    print(f"\n2. Colonnes avec >50% NaN: {cols_high_nan}")
    for col in cols_high_nan:
        print(f"   - {col}: remplissage avec -1")
        gold_df[col] = gold_df[col].fillna(-1)

# 3. Interpolation pour colonnes temporelles (pressure, temp)
cols_to_interpolate = [col for col in gold_df.columns 
                       if any(x in col for x in ['pressure', 'temp', 'trend'])]
cols_to_interpolate = [col for col in cols_to_interpolate if gold_df[col].isna().sum() > 0]
if cols_to_interpolate:
    print(f"\n3. Interpolation pour colonnes temporelles: {cols_to_interpolate}")
    for col in cols_to_interpolate:
        gold_df[col] = gold_df[col].interpolate(method='linear', limit_direction='both')

# 4. Remplissage avec médiane pour colonnes restantes
remaining_nan = gold_df.columns[gold_df.isna().any()].tolist()
if remaining_nan:
    print(f"\n4. Remplissage avec médiane: {remaining_nan}")
    for col in remaining_nan:
        median_val = gold_df[col].median()
        gold_df[col] = gold_df[col].fillna(median_val)
        print(f"   - {col}: {gold_df[col].isna().sum()} NaN restants")

# 5. Vérification finale
total_nan = gold_df.isna().sum().sum()
print(f"\n✓ RÉSULTAT FINAL: {total_nan} NaN restants")
print(f"Dimensions: {gold_df.shape}")


=== NETTOYAGE DES NaN ===

1. Suppression de colonnes vides (100% NaN): ['ambient_humidity_pct']

2. Colonnes avec >50% NaN: ['incident_max_severity_prev_24h']
   - incident_max_severity_prev_24h: remplissage avec -1

3. Interpolation pour colonnes temporelles: ['temp_std_6h', 'pressure_mean_6h', 'pressure_max_6h', 'pressure_std_6h', 'temp_std_12h', 'pressure_mean_12h', 'pressure_max_12h', 'pressure_std_12h', 'temp_std_24h', 'pressure_mean_24h', 'pressure_max_24h', 'pressure_std_24h', 'temp_trend_6h', 'pressure_trend_6h', 'temp_zscore_24h']

4. Remplissage avec médiane: ['hours_since_last_incident']
   - hours_since_last_incident: 0 NaN restants

✓ RÉSULTAT FINAL: 0 NaN restants
Dimensions: (66678, 44)


In [135]:
# ===== ANALYSE CORRELATION ENTRE PRESSURE ET NaN =====

# Recharger le dataset original pour avoir les NaN
gold_df_original = pd.read_csv("C:\\Formation\\gold_dataset\\gold_dataset_20260526-104215.csv")

print("=== CORRÉLATION PRESSURE & NaN ===\n")

# 1. Identifier les colonnes pressure avec NaN
pressure_cols = [col for col in gold_df_original.columns if 'pressure' in col]
pressure_cols_with_nan = [col for col in pressure_cols if gold_df_original[col].isna().sum() > 0]

print(f"Colonnes pressure avec NaN: {pressure_cols_with_nan}\n")

# 2. Créer une colonne indicatrice: 1 si NaN dans pressure, 0 sinon
gold_df_original['has_pressure_nan'] = gold_df_original[pressure_cols_with_nan].isna().any(axis=1).astype(int)

# 3. Analyser les patterns des NaN
print(f"Nombre de lignes avec NaN dans pressure: {gold_df_original['has_pressure_nan'].sum()}")
print(f"Pourcentage: {gold_df_original['has_pressure_nan'].sum() / len(gold_df_original) * 100:.2f}%\n")

# 4. Vérifier si les NaN pressure coincident avec NaN d'autres colonnes
print("Corrélation entre NaN pressure et NaN d'autres colonnes:")
cols_nan_correlation = []
for col in gold_df_original.columns:
    if col not in pressure_cols and gold_df_original[col].isna().sum() > 0:
        # Comparer les lignes avec NaN
        both_nan = ((gold_df_original[col].isna() & gold_df_original['has_pressure_nan'].astype(bool)).sum())
        if both_nan > 0:
            pct = both_nan / gold_df_original['has_pressure_nan'].sum() * 100
            cols_nan_correlation.append((col, both_nan, pct))

if cols_nan_correlation:
    for col, count, pct in sorted(cols_nan_correlation, key=lambda x: x[2], reverse=True):
        print(f"  {col}: {count} lignes ({pct:.1f}% des lignes avec pressure NaN)")
else:
    print("  Aucune corrélation détectée - NaN pressure sont indépendants")

# 5. Vérifier si c'est un pattern temporel (lignes consécutives)
nan_indices = gold_df_original[gold_df_original['has_pressure_nan'] > 0].index.values
if len(nan_indices) > 0:
    print(f"\nPattern temporel des NaN pressure:")
    print(f"  Première occurrence: index {nan_indices[0]}")
    print(f"  Dernière occurrence: index {nan_indices[-1]}")
    gaps = nan_indices[1:] - nan_indices[:-1]
    print(f"  Écart moyen entre NaN: {gaps.mean():.0f} lignes")
    print(f"  Écart max: {gaps.max()} lignes")


=== CORRÉLATION PRESSURE & NaN ===

Colonnes pressure avec NaN: ['pressure_mean_6h', 'pressure_max_6h', 'pressure_std_6h', 'pressure_mean_12h', 'pressure_max_12h', 'pressure_std_12h', 'pressure_mean_24h', 'pressure_max_24h', 'pressure_std_24h', 'pressure_trend_6h']

Nombre de lignes avec NaN dans pressure: 1504
Pourcentage: 2.26%

Corrélation entre NaN pressure et NaN d'autres colonnes:
  ambient_humidity_pct: 1504 lignes (100.0% des lignes avec pressure NaN)
  incident_max_severity_prev_24h: 1396 lignes (92.8% des lignes avec pressure NaN)
  hours_since_last_incident: 347 lignes (23.1% des lignes avec pressure NaN)
  temp_trend_6h: 90 lignes (6.0% des lignes avec pressure NaN)
  temp_std_6h: 15 lignes (1.0% des lignes avec pressure NaN)
  temp_std_12h: 15 lignes (1.0% des lignes avec pressure NaN)
  temp_std_24h: 15 lignes (1.0% des lignes avec pressure NaN)
  temp_zscore_24h: 15 lignes (1.0% des lignes avec pressure NaN)

Pattern temporel des NaN pressure:
  Première occurrence: inde

# ===== SÉPARATION TRAIN / TEST =====

# Colonnes à exclure des features
COLS_META = ['machine_code', 'window_start', 'window_end', 'split_set']
COLS_LABELS = ['label_failure_next_6h', 'label_failure_next_12h',
               'label_failure_next_24h', 'label_failure_next_48h']

In [136]:
# ===== SÉPARATION TRAIN / TEST =====
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, accuracy_score, precision_score, recall_score, f1_score

COLS_META   = ['machine_code', 'window_start', 'window_end', 'split_set']
COLS_LABELS = ['label_failure_next_6h', 'label_failure_next_12h',
               'label_failure_next_24h', 'label_failure_next_48h']

feature_cols = [c for c in gold_df.columns if c not in COLS_META + COLS_LABELS]
TARGET = 'label_failure_next_24h'

train_df = gold_df[gold_df['split_set'] == 'train']
test_df  = gold_df[gold_df['split_set'] == 'test']

X_train = train_df[feature_cols]
y_train = train_df[TARGET].astype(int)
X_test  = test_df[feature_cols]
y_test  = test_df[TARGET].astype(int)

print(f"Features : {len(feature_cols)}")
print(f"Train    : {X_train.shape[0]} lignes  |  positifs: {y_train.sum()} ({y_train.mean()*100:.1f}%)")
print(f"Test     : {X_test.shape[0]} lignes  |  positifs: {y_test.sum()} ({y_test.mean()*100:.1f}%)")

# ===== ENTRAINEMENT - RANDOM FOREST =====
rf_params = dict(n_estimators=100, max_depth=10, class_weight='balanced', random_state=42, n_jobs=-1)

with mlflow.start_run(run_name="RandomForest"):
    mlflow.log_params(rf_params)
    mlflow.log_param("target", TARGET)

    model = RandomForestClassifier(**rf_params)
    model.fit(X_train, y_train)

    y_pred  = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    rf_pred, rf_proba = y_pred, y_proba

    rf_confusion = confusion_matrix(y_test, y_pred)
    rf_auc       = roc_auc_score(y_test, y_proba)
    rf_accuracy  = accuracy_score(y_test, y_pred)
    rf_precision = precision_score(y_test, y_pred, zero_division=0)
    rf_recall    = recall_score(y_test, y_pred, zero_division=0)
    rf_f1        = f1_score(y_test, y_pred, zero_division=0)

    mlflow.log_metric("accuracy",  rf_accuracy)
    mlflow.log_metric("precision", rf_precision)
    mlflow.log_metric("recall",    rf_recall)
    mlflow.log_metric("f1",        rf_f1)
    mlflow.log_metric("auc_roc",   rf_auc)
    mlflow.sklearn.log_model(model, "model")

print(classification_report(y_test, y_pred, target_names=['Pas de panne', 'Panne']))
print(f"AUC-ROC : {rf_auc:.4f}")
print("\nMatrice de confusion :")
print(rf_confusion)

Features : 36
Train    : 46663 lignes  |  positifs: 2393 (5.1%)
Test     : 10010 lignes  |  positifs: 336 (3.4%)


2026/05/27 13:13:08 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/27 13:13:08 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


              precision    recall  f1-score   support

Pas de panne       0.96      0.79      0.87      9674
       Panne       0.02      0.14      0.04       336

    accuracy                           0.76     10010
   macro avg       0.49      0.46      0.45     10010
weighted avg       0.93      0.76      0.84     10010

AUC-ROC : 0.4610

Matrice de confusion :
[[7600 2074]
 [ 290   46]]


In [ ]:
# ===== RANDOM FOREST OPTIMISÉ — MAXIMISER TP / MINIMISER FN =====
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import precision_recall_curve, fbeta_score, make_scorer
from joblib import parallel_backend
import numpy as np

X_train_np = X_train.to_numpy().astype(float)
y_train_np = y_train.to_numpy().astype(int)
X_test_np  = X_test.to_numpy().astype(float)

# F2 : recall pèse 2× la précision → favorise TP sans tout prédire positif
f2_scorer = make_scorer(fbeta_score, beta=2, zero_division=0)

pos_ratio = (y_train == 0).sum() / (y_train == 1).sum()
param_dist_rf = {
    'n_estimators':      [100, 200, 300, 500],
    'max_depth':         [5, 10, 15, 20, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf':  [1, 2, 4],
    'max_features':      ['sqrt', 'log2', None],
    'class_weight': [
        'balanced',
        {0: 1, 1: int(pos_ratio)},
        {0: 1, 1: int(pos_ratio * 1.5)},
        {0: 1, 1: int(pos_ratio * 2)},
        {0: 1, 1: int(pos_ratio * 3)},
    ],
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

rf_search = RandomizedSearchCV(
    estimator=RandomForestClassifier(random_state=42, n_jobs=-1),
    param_distributions=param_dist_rf,
    n_iter=40,
    scoring=f2_scorer,
    cv=cv,
    verbose=1,
    random_state=42,
    n_jobs=1
)

with mlflow.start_run(run_name="RandomForest-tuned"):
    with parallel_backend('threading', n_jobs=-1):
        rf_search.fit(X_train_np, y_train_np)

    rf_tuned       = rf_search.best_estimator_
    rf_tuned_proba = rf_tuned.predict_proba(X_test_np)[:, 1]

    precisions, recalls, thresholds = precision_recall_curve(y_test, rf_tuned_proba)
    f2_scores_thresh = (1 + 2**2) * precisions * recalls / (2**2 * precisions + recalls + 1e-9)
    threshold_f2     = float(thresholds[np.argmax(f2_scores_thresh[:-1])])
    f1_scores_thresh = 2 * precisions * recalls / (precisions + recalls + 1e-9)
    threshold_f1     = float(thresholds[np.argmax(f1_scores_thresh[:-1])])

    rf_tuned_pred = (rf_tuned_proba >= threshold_f2).astype(int)

    rf_tuned_confusion = confusion_matrix(y_test, rf_tuned_pred)
    rf_tuned_auc       = roc_auc_score(y_test, rf_tuned_proba)
    rf_tuned_accuracy  = accuracy_score(y_test, rf_tuned_pred)
    rf_tuned_precision = precision_score(y_test, rf_tuned_pred, zero_division=0)
    rf_tuned_recall    = recall_score(y_test, rf_tuned_pred, zero_division=0)
    rf_tuned_f1        = f1_score(y_test, rf_tuned_pred, zero_division=0)

    tn, fp, fn, tp = rf_tuned_confusion.ravel()

    mlflow.log_params(rf_search.best_params_)
    mlflow.log_param("target",       TARGET)
    mlflow.log_param("threshold_f2", round(threshold_f2, 4))
    mlflow.log_param("threshold_f1", round(threshold_f1, 4))
    mlflow.log_metric("best_f2_cv",  rf_search.best_score_)
    mlflow.log_metric("TP",          int(tp))
    mlflow.log_metric("FN",          int(fn))
    mlflow.log_metric("FP",          int(fp))
    mlflow.log_metric("TN",          int(tn))
    mlflow.log_metric("accuracy",    rf_tuned_accuracy)
    mlflow.log_metric("precision",   rf_tuned_precision)
    mlflow.log_metric("recall",      rf_tuned_recall)
    mlflow.log_metric("f1",          rf_tuned_f1)
    mlflow.log_metric("auc_roc",     rf_tuned_auc)
    mlflow.sklearn.log_model(rf_tuned, "model")

print(f"Best params    : {rf_search.best_params_}")
print(f"Best F2 CV     : {rf_search.best_score_:.4f}")
print(f"Seuil F2       : {threshold_f2:.4f}  → recall×2 vs précision")
print(f"Seuil F1       : {threshold_f1:.4f}  → compromis équilibré")
print(classification_report(y_test, rf_tuned_pred, target_names=['Pas de panne', 'Panne']))
print(f"AUC-ROC   : {rf_tuned_auc:.4f}")
print(rf_tuned_confusion)
print(f"\nTP : {tp}  FN : {fn}  FP : {fp}  ({tp/(tp+fn)*100:.1f}% pannes détectées)")
print(f"Precision : {rf_tuned_precision:.4f}  Recall : {rf_tuned_recall:.4f}  F1 : {rf_tuned_f1:.4f}")

lass_weight='balanced' — compense le déséquilibre (peu de pannes vs beaucoup de lignes normales)
AUC-ROC — métrique adaptée aux classes déséquilibrées, meilleure que l'accuracy
n_jobs=-1 — utilise tous les cœurs CPU
Une fois ce premier modèle évalué, on peut passer à XGBoost ou LightGBM pour de meilleures performances.

## Avec la Régression Logistique 

In [138]:
# ===== ENTRAINEMENT - RÉGRESSION LOGISTIQUE =====
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

lr_params = dict(class_weight='balanced', max_iter=1000, random_state=42)

with mlflow.start_run(run_name="LogisticRegression"):
    mlflow.log_params(lr_params)
    mlflow.log_param("target",  TARGET)
    mlflow.log_param("scaler",  "StandardScaler")

    model = LogisticRegression(**lr_params)
    model.fit(X_train_scaled, y_train)

    y_pred  = model.predict(X_test_scaled)
    y_proba = model.predict_proba(X_test_scaled)[:, 1]
    lr_pred, lr_proba = y_pred, y_proba

    lr_confusion = confusion_matrix(y_test, y_pred)
    lr_auc       = roc_auc_score(y_test, y_proba)
    lr_accuracy  = accuracy_score(y_test, y_pred)
    lr_precision = precision_score(y_test, y_pred, zero_division=0)
    lr_recall    = recall_score(y_test, y_pred, zero_division=0)
    lr_f1        = f1_score(y_test, y_pred, zero_division=0)

    mlflow.log_metric("accuracy",  lr_accuracy)
    mlflow.log_metric("precision", lr_precision)
    mlflow.log_metric("recall",    lr_recall)
    mlflow.log_metric("f1",        lr_f1)
    mlflow.log_metric("auc_roc",   lr_auc)
    mlflow.sklearn.log_model(model, "model")

print(classification_report(y_test, y_pred, target_names=['Pas de panne', 'Panne']))
print(f"AUC-ROC : {lr_auc:.4f}")
print("\nMatrice de confusion :")
print(lr_confusion)

feature_importance = pd.DataFrame({
    'feature':     feature_cols,
    'coefficient': model.coef_[0]
}).sort_values('coefficient', key=abs, ascending=False)
print("\nTop 10 features influentes :")
print(feature_importance.head(10))

2026/05/27 13:30:39 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/27 13:30:39 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


              precision    recall  f1-score   support

Pas de panne       0.97      0.31      0.47      9674
       Panne       0.03      0.69      0.06       336

    accuracy                           0.32     10010
   macro avg       0.50      0.50      0.26     10010
weighted avg       0.93      0.32      0.45     10010

AUC-ROC : 0.4931

Matrice de confusion :
[[2971 6703]
 [ 104  232]]

Top 10 features influentes :
                    feature  coefficient
15        pressure_mean_24h    -3.342202
10         pressure_max_12h     2.561309
14             temp_std_24h    -1.037475
16         pressure_max_24h     0.944669
13             temp_max_24h     0.725326
12            temp_mean_24h    -0.443131
3          pressure_mean_6h    -0.401393
0              temp_mean_6h    -0.349576
34           ambient_temp_c    -0.295141
21  incident_count_prev_24h     0.218725


In [ ]:
# ===== RÉGRESSION LOGISTIQUE OPTIMISÉE — MAXIMISER TP / MINIMISER FN =====
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import precision_recall_curve, fbeta_score, make_scorer
from joblib import parallel_backend
import numpy as np

X_train_lr = X_train_scaled.astype(float)
y_train_lr = y_train.to_numpy().astype(int)
X_test_lr  = X_test_scaled.astype(float)

# F2 : recall pèse 2× la précision → favorise TP sans tout prédire positif
f2_scorer = make_scorer(fbeta_score, beta=2, zero_division=0)

pos_ratio = (y_train == 0).sum() / (y_train == 1).sum()
param_dist_lr = {
    'C': [0.001, 0.005, 0.01, 0.05, 0.1, 0.5, 1, 5],
    'class_weight': [
        'balanced',
        {0: 1, 1: int(pos_ratio)},
        {0: 1, 1: int(pos_ratio * 1.5)},
        {0: 1, 1: int(pos_ratio * 2)},
        {0: 1, 1: int(pos_ratio * 3)},
    ],
    'penalty':  ['l1', 'l2'],
    'solver':   ['saga'],
    'max_iter': [3000],
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

lr_search = RandomizedSearchCV(
    estimator=LogisticRegression(random_state=42),
    param_distributions=param_dist_lr,
    n_iter=40,
    scoring=f2_scorer,
    cv=cv,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

with mlflow.start_run(run_name="LogisticRegression-tuned"):
    with parallel_backend('threading', n_jobs=-1):
        lr_search.fit(X_train_lr, y_train_lr)

    lr_tuned       = lr_search.best_estimator_
    lr_tuned_proba = lr_tuned.predict_proba(X_test_lr)[:, 1]

    precisions, recalls, thresholds = precision_recall_curve(y_test, lr_tuned_proba)
    f2_scores_thresh = (1 + 2**2) * precisions * recalls / (2**2 * precisions + recalls + 1e-9)
    threshold_f2     = float(thresholds[np.argmax(f2_scores_thresh[:-1])])
    f1_scores_thresh = 2 * precisions * recalls / (precisions + recalls + 1e-9)
    threshold_f1     = float(thresholds[np.argmax(f1_scores_thresh[:-1])])

    lr_tuned_pred = (lr_tuned_proba >= threshold_f2).astype(int)

    lr_tuned_confusion = confusion_matrix(y_test, lr_tuned_pred)
    lr_tuned_auc       = roc_auc_score(y_test, lr_tuned_proba)
    lr_tuned_accuracy  = accuracy_score(y_test, lr_tuned_pred)
    lr_tuned_precision = precision_score(y_test, lr_tuned_pred, zero_division=0)
    lr_tuned_recall    = recall_score(y_test, lr_tuned_pred, zero_division=0)
    lr_tuned_f1        = f1_score(y_test, lr_tuned_pred, zero_division=0)

    tn, fp, fn, tp = lr_tuned_confusion.ravel()

    mlflow.log_params(lr_search.best_params_)
    mlflow.log_param("target",       TARGET)
    mlflow.log_param("threshold_f2", round(threshold_f2, 4))
    mlflow.log_param("threshold_f1", round(threshold_f1, 4))
    mlflow.log_metric("best_f2_cv",  lr_search.best_score_)
    mlflow.log_metric("TP",          int(tp))
    mlflow.log_metric("FN",          int(fn))
    mlflow.log_metric("FP",          int(fp))
    mlflow.log_metric("TN",          int(tn))
    mlflow.log_metric("accuracy",    lr_tuned_accuracy)
    mlflow.log_metric("precision",   lr_tuned_precision)
    mlflow.log_metric("recall",      lr_tuned_recall)
    mlflow.log_metric("f1",          lr_tuned_f1)
    mlflow.log_metric("auc_roc",     lr_tuned_auc)
    mlflow.sklearn.log_model(lr_tuned, "model")

print(f"Best params    : {lr_search.best_params_}")
print(f"Best F2 CV     : {lr_search.best_score_:.4f}")
print(f"Seuil F2       : {threshold_f2:.4f}  → recall×2 vs précision")
print(f"Seuil F1       : {threshold_f1:.4f}  → compromis équilibré")
print(classification_report(y_test, lr_tuned_pred, target_names=['Pas de panne', 'Panne']))
print(f"AUC-ROC   : {lr_tuned_auc:.4f}")
print(lr_tuned_confusion)
print(f"\nTP : {tp}  FN : {fn}  FP : {fp}  ({tp/(tp+fn)*100:.1f}% pannes détectées)")
print(f"Precision : {lr_tuned_precision:.4f}  Recall : {lr_tuned_recall:.4f}  F1 : {lr_tuned_f1:.4f}")

## Avec le modèle XGBoost

Avantages XGBoost :

scale_pos_weight gère automatiquement le déséquilibre
Pas besoin de normalisation (contrairement à la régression logistique)
Généralement plus performant que Random Forest et régression logistique
Feature importance plus fiable

In [140]:
# ===== ENTRAINEMENT - XGBOOST BASELINE =====
import xgboost as xgb

scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

xgb_params = dict(
    n_estimators=200, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    random_state=42, n_jobs=-1, verbosity=0
)

with mlflow.start_run(run_name="XGBoost-baseline"):
    mlflow.log_params(xgb_params)
    mlflow.log_param("target", TARGET)

    model = xgb.XGBClassifier(**xgb_params)
    model.fit(X_train, y_train)

    y_pred  = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    xgb_pred, xgb_proba = y_pred, y_proba

    xgb_confusion = confusion_matrix(y_test, y_pred)
    xgb_auc       = roc_auc_score(y_test, y_proba)
    xgb_accuracy  = accuracy_score(y_test, y_pred)
    xgb_precision = precision_score(y_test, y_pred, zero_division=0)
    xgb_recall    = recall_score(y_test, y_pred, zero_division=0)
    xgb_f1        = f1_score(y_test, y_pred, zero_division=0)

    mlflow.log_metric("accuracy",  xgb_accuracy)
    mlflow.log_metric("precision", xgb_precision)
    mlflow.log_metric("recall",    xgb_recall)
    mlflow.log_metric("f1",        xgb_f1)
    mlflow.log_metric("auc_roc",   xgb_auc)
    mlflow.xgboost.log_model(model, "model")

print(classification_report(y_test, y_pred, target_names=['Pas de panne', 'Panne']))
print(f"AUC-ROC : {xgb_auc:.4f}")
print("\nMatrice de confusion :")
print(xgb_confusion)

feature_importance = pd.DataFrame({
    'feature':    feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)
print("\nTop 10 features importantes :")
print(feature_importance.head(10))

2026/05/27 14:00:38 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


              precision    recall  f1-score   support

Pas de panne       0.97      0.92      0.94      9674
       Panne       0.04      0.09      0.05       336

    accuracy                           0.89     10010
   macro avg       0.50      0.50      0.50     10010
weighted avg       0.94      0.89      0.91     10010

AUC-ROC : 0.5017

Matrice de confusion :
[[8926  748]
 [ 307   29]]

Top 10 features importantes :
                      feature  importance
34             ambient_temp_c    0.061070
16           pressure_max_24h    0.059804
13               temp_max_24h    0.057662
23     incident_count_prev_7d    0.054102
24  hours_since_last_incident    0.050715
12              temp_mean_24h    0.048865
35       ambient_pressure_hpa    0.047663
21    incident_count_prev_24h    0.046577
17           pressure_std_24h    0.045732
10           pressure_max_12h    0.045041


In [141]:
# ===== TUNING XGBOOST OPTIMISÉ POUR F1/RECALL =====
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import precision_recall_curve
from joblib import parallel_backend
import numpy as np

# Conversion numpy : évite les StringDtype pandas 3.x non-sérialisables
X_train_np = X_train.to_numpy().astype(float)
y_train_np = y_train.to_numpy().astype(int)
X_test_np  = X_test.to_numpy().astype(float)

param_dist = {
    'max_depth':        [3, 4, 5, 6],
    'learning_rate':    [0.01, 0.02, 0.05, 0.1],
    'n_estimators':     [200, 300, 400, 500],
    'subsample':        [0.6, 0.7, 0.8, 0.9],
    'colsample_bytree': [0.5, 0.6, 0.7, 0.8],
    'min_child_weight': [1, 2, 3, 5],
    'gamma':            [0, 0.05, 0.1, 0.2],
    'scale_pos_weight': [scale_pos_weight, scale_pos_weight * 1.5,
                         scale_pos_weight * 2, scale_pos_weight * 3],
    'reg_alpha':        [0, 0.01, 0.1],
    'reg_lambda':       [0.5, 1.0, 1.5]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

xgb_search = RandomizedSearchCV(
    estimator=xgb.XGBClassifier(
        random_state=42, n_jobs=-1, verbosity=0, eval_metric='aucpr'
    ),
    param_distributions=param_dist,
    n_iter=50,
    scoring='f1',
    cv=cv,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

with mlflow.start_run(run_name="XGBoost-tuned"):
    # backend threading : threads partagent la mémoire → pas de pickling → résout le BrokenProcessPool
    with parallel_backend('threading', n_jobs=-1):
        xgb_search.fit(X_train_np, y_train_np)

    xgb_tuned       = xgb_search.best_estimator_
    xgb_tuned_proba = xgb_tuned.predict_proba(X_test_np)[:, 1]

    # Seuil optimal sur la courbe precision-recall
    precisions, recalls, thresholds = precision_recall_curve(y_test, xgb_tuned_proba)
    f1_scores_thresh = 2 * precisions * recalls / (precisions + recalls + 1e-9)
    best_idx       = np.argmax(f1_scores_thresh)
    best_threshold = float(thresholds[best_idx])

    xgb_tuned_pred = (xgb_tuned_proba >= best_threshold).astype(int)

    xgb_tuned_confusion = confusion_matrix(y_test, xgb_tuned_pred)
    xgb_tuned_auc       = roc_auc_score(y_test, xgb_tuned_proba)
    xgb_tuned_accuracy  = accuracy_score(y_test, xgb_tuned_pred)
    xgb_tuned_precision = precision_score(y_test, xgb_tuned_pred, zero_division=0)
    xgb_tuned_recall    = recall_score(y_test, xgb_tuned_pred, zero_division=0)
    xgb_tuned_f1        = f1_score(y_test, xgb_tuned_pred, zero_division=0)

    mlflow.log_params(xgb_search.best_params_)
    mlflow.log_param("target",    TARGET)
    mlflow.log_param("threshold", round(best_threshold, 4))
    mlflow.log_metric("best_f1_cv",  xgb_search.best_score_)
    mlflow.log_metric("accuracy",    xgb_tuned_accuracy)
    mlflow.log_metric("precision",   xgb_tuned_precision)
    mlflow.log_metric("recall",      xgb_tuned_recall)
    mlflow.log_metric("f1",          xgb_tuned_f1)
    mlflow.log_metric("auc_roc",     xgb_tuned_auc)
    mlflow.xgboost.log_model(xgb_tuned, "model")

print(f"Seuil optimal  : {best_threshold:.4f}  (défaut = 0.5)")
print(f"Best F1 CV     : {xgb_search.best_score_:.4f}")
print('\n=== XGBoost optimisé — seuil ajusté ===')
print(classification_report(y_test, xgb_tuned_pred, target_names=['Pas de panne', 'Panne']))
print(f"AUC-ROC : {xgb_tuned_auc:.4f}")
print('\nMatrice de confusion :')
print(xgb_tuned_confusion)
print(f"\nAccuracy  : {xgb_tuned_accuracy:.4f}")
print(f"Precision : {xgb_tuned_precision:.4f}")
print(f"Recall    : {xgb_tuned_recall:.4f}")
print(f"F1-score  : {xgb_tuned_f1:.4f}")

Fitting 5 folds for each of 50 candidates, totalling 250 fits


2026/05/27 14:07:15 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Seuil optimal  : 0.0010  (défaut = 0.5)
Best F1 CV     : 0.9240

=== XGBoost optimisé — seuil ajusté ===
              precision    recall  f1-score   support

Pas de panne       0.98      0.21      0.34      9674
       Panne       0.04      0.85      0.07       336

    accuracy                           0.23     10010
   macro avg       0.51      0.53      0.21     10010
weighted avg       0.94      0.23      0.33     10010

AUC-ROC : 0.5046

Matrice de confusion :
[[2018 7656]
 [  50  286]]

Accuracy  : 0.2302
Precision : 0.0360
Recall    : 0.8512
F1-score  : 0.0691


In [142]:
# ===== COMPARAISON DES MATRICES DE CONFUSION ET MÉTRIQUES =====

confusion_comparison = pd.DataFrame([
    {
        'Modèle': 'Random Forest',
        'TN': rf_confusion[0, 0],
        'FP': rf_confusion[0, 1],
        'FN': rf_confusion[1, 0],
        'TP': rf_confusion[1, 1],
        'AUC-ROC': rf_auc,
        'Accuracy': rf_accuracy,
        'Precision': rf_precision,
        'Recall': rf_recall,
        'F1-score': rf_f1
    },
    {
        'Modèle': 'Régression Logistique',
        'TN': lr_confusion[0, 0],
        'FP': lr_confusion[0, 1],
        'FN': lr_confusion[1, 0],
        'TP': lr_confusion[1, 1],
        'AUC-ROC': lr_auc,
        'Accuracy': lr_accuracy,
        'Precision': lr_precision,
        'Recall': lr_recall,
        'F1-score': lr_f1
    },
    {
        'Modèle': 'XGBoost',
        'TN': xgb_confusion[0, 0],
        'FP': xgb_confusion[0, 1],
        'FN': xgb_confusion[1, 0],
        'TP': xgb_confusion[1, 1],
        'AUC-ROC': xgb_auc,
        'Accuracy': xgb_accuracy,
        'Precision': xgb_precision,
        'Recall': xgb_recall,
        'F1-score': xgb_f1
    }
])

print(confusion_comparison.to_string(index=False))

               Modèle   TN   FP  FN  TP  AUC-ROC  Accuracy  Precision   Recall  F1-score
        Random Forest 7600 2074 290  46 0.461006  0.763836   0.021698 0.136905  0.037459
Régression Logistique 2971 6703 104 232 0.493069  0.319980   0.033453 0.690476  0.063815
              XGBoost 8926  748 307  29 0.501721  0.894605   0.037323 0.086310  0.052111


In [143]:
# ===== TABLEAU DE SYNTHÈSE DES MÉTRIQUES =====

metric_summary = confusion_comparison[[
    'Modèle',
    'Accuracy',
    'Precision',
    'Recall',
    'F1-score'
]]

print(metric_summary.to_string(index=False))

               Modèle  Accuracy  Precision   Recall  F1-score
        Random Forest  0.763836   0.021698 0.136905  0.037459
Régression Logistique  0.319980   0.033453 0.690476  0.063815
              XGBoost  0.894605   0.037323 0.086310  0.052111


La méthode de régression logistique semble être le meilleur modèle avec le meilleur Recall et F1 score ainsi que le meilleur TP et le plus faible FN.

In [144]:
# ===== COMPARAISON DES PRÉDICTIONS =====

prediction_comparison = pd.DataFrame([
    {
        'Modèle': 'Random Forest',
        'Prédits 0': int((rf_pred == 0).sum()),
        'Prédits 1': int((rf_pred == 1).sum()),
        'Taux positifs prédits (%)': float((rf_pred == 1).mean() * 100)
    },
    {
        'Modèle': 'Régression Logistique',
        'Prédits 0': int((lr_pred == 0).sum()),
        'Prédits 1': int((lr_pred == 1).sum()),
        'Taux positifs prédits (%)': float((lr_pred == 1).mean() * 100)
    },
    {
        'Modèle': 'XGBoost',
        'Prédits 0': int((xgb_pred == 0).sum()),
        'Prédits 1': int((xgb_pred == 1).sum()),
        'Taux positifs prédits (%)': float((xgb_pred == 1).mean() * 100)
    }
])

print(prediction_comparison.to_string(index=False))

               Modèle  Prédits 0  Prédits 1  Taux positifs prédits (%)
        Random Forest       7890       2120                  21.178821
Régression Logistique       3075       6935                  69.280719
              XGBoost       9233        777                   7.762238


Pour la prédiction de pannes, le meilleur modèle est la **régression logistique**.

### Justification précise
- Cible utilisée : `label_failure_next_24h`
- Importance : pour prédire les pannes, on veut surtout maximiser le **recall** (capturer le plus de vraies pannes possible) et avoir un bon **F1-score**.

### Résultats comparés
- Régression logistique
  - Accuracy : `0.31998`
  - Precision : `0.03345`
  - Recall : `0.69048`
  - F1-score : `0.06382`
- XGBoost
  - Accuracy : `0.89460`
  - Precision : `0.03732`
  - Recall : `0.08631`
  - F1-score : `0.05211`
- Random Forest
  - Accuracy : `0.76384`
  - Precision : `0.02170`
  - Recall : `0.13691`
  - F1-score : `0.03746`

### Pourquoi la régression logistique est meilleure
- Elle a le **meilleur recall** : `0.69048`, donc elle détecte beaucoup plus de vraies pannes.
- Elle a aussi le **meilleur F1-score** : `0.06382`, ce qui signifie un meilleur compromis entre precision et recall.
- Les autres modèles ont des recalls très faibles, donc ils manquent beaucoup de pannes malgré une accuracy plus élevée.

> Conclusion : pour la détection de pannes, la métrique la plus pertinente est le recall, et la régression logistique est le meilleur modèle selon cette métrique et selon le F1-score.

Optimiser la méthode de régression logistique pour améliorer le recall sur la détection de panne.

Ce qui est optimisé :

Levier	Pourquoi
C (0.001 → 10)	: Contrôle la régularisation — une valeur faible force le modèle à généraliser, ce qui peut améliorer le recall sur la classe minoritaire
class_weight custom	 : 'balanced' pondère selon la fréquence, mais on teste aussi des poids explicitement plus agressifs (×1.5, ×2, ×3 du ratio réel) pour forcer la détection des pannes
solver lbfgs/saga : saga supporte mieux les grands datasets avec l1/l2
Seuil optimal : Même approche que XGBoost — la courbe precision-recall trouve le seuil qui maximise le F1

Voici ce qui change par rapport à la version précédente :

Avant	Maintenant
scoring	'f1'	'recall' — pousse le modèle à détecter plus de pannes
class_weight	jusqu'à ×3	jusqu'à ×10 du ratio réel — pénalise fortement les FN
penalty	l2 seul	l1 + l2 — l1 peut éliminer les features bruitées et améliorer le recall
solver	lbfgs/saga	saga uniquement — seul solver qui supporte l1 et l2 sur grands datasets
Seuil	un seul (max F1)	deux seuils : threshold_recall (max TP/min FN) et threshold_f1 (compromis)
Sortie	métriques génériques	TP, FN, FP explicites avec pourcentages de pannes détectées

Cellule remplacée. Voici ce qui change par rapport à la version précédente :

Cellule RandomForest-tuned insérée juste après le baseline RF. Elle est identique dans sa structure à l'optimisation LR :
Paramètre optimisé	Plage testée
n_estimators	100 → 500
max_depth	5, 10, 15, 20, None
min_samples_split / min_samples_leaf	granularité des feuilles
max_features	sqrt, log2, None
class_weight	ratio réel × 1, 2, 3, 5, 8, 10
scoring='recall' + deux seuils (max-recall avec précision ≥ 3%, max-F1) + TP/FN explicites dans la sortie. Le run apparaîtra dans MLflow sous RandomForest-tuned.



In [145]:
# ===== TABLEAU DE SYNTHÈSE — TOUS LES MODÈLES =====
import pandas as pd

def model_row(name, confusion, accuracy, precision, recall, f1, auc):
    tn, fp, fn, tp = confusion.ravel()
    total_pos = int(tp + fn)
    return {
        'Modèle':          name,
        'TP':              int(tp),
        'FN':              int(fn),
        'FP':              int(fp),
        'TN':              int(tn),
        '% pannes détectées': f"{tp/total_pos*100:.1f}%",
        '% pannes manquées':  f"{fn/total_pos*100:.1f}%",
        'Recall':          round(recall,    4),
        'Precision':       round(precision, 4),
        'F1-score':        round(f1,        4),
        'AUC-ROC':         round(auc,       4),
        'Accuracy':        round(accuracy,  4),
    }

rows = []

# --- Baselines
rows.append(model_row("RF baseline",  rf_confusion,  rf_accuracy,  rf_precision,  rf_recall,  rf_f1,  rf_auc))
rows.append(model_row("LR baseline",  lr_confusion,  lr_accuracy,  lr_precision,  lr_recall,  lr_f1,  lr_auc))
rows.append(model_row("XGB baseline", xgb_confusion, xgb_accuracy, xgb_precision, xgb_recall, xgb_f1, xgb_auc))

# --- Optimisés (disponibles uniquement si les cellules ont été exécutées)
try:
    rows.append(model_row("RF tuned",  rf_tuned_confusion,  rf_tuned_accuracy,  rf_tuned_precision,  rf_tuned_recall,  rf_tuned_f1,  rf_tuned_auc))
except NameError:
    pass

try:
    rows.append(model_row("LR tuned",  lr_tuned_confusion,  lr_tuned_accuracy,  lr_tuned_precision,  lr_tuned_recall,  lr_tuned_f1,  lr_tuned_auc))
except NameError:
    pass

try:
    rows.append(model_row("XGB tuned", xgb_tuned_confusion, xgb_tuned_accuracy, xgb_tuned_precision, xgb_tuned_recall, xgb_tuned_f1, xgb_tuned_auc))
except NameError:
    pass

synthese = pd.DataFrame(rows)

# Mise en forme : tri par recall décroissant
synthese = synthese.sort_values('Recall', ascending=False).reset_index(drop=True)

print("=" * 90)
print("SYNTHÈSE — COMPARAISON DES MODÈLES  (trié par Recall décroissant)")
print("=" * 90)
print(synthese.to_string(index=False))
print()

# Identifier le meilleur modèle selon chaque critère
print("--- Meilleur par critère ---")
for col in ['Recall', 'F1-score', 'Precision', 'AUC-ROC', 'TP']:
    best = synthese.loc[synthese[col].idxmax(), 'Modèle']
    val  = synthese[col].max()
    print(f"  {col:<20}: {best}  ({val})")

SYNTHÈSE — COMPARAISON DES MODÈLES  (trié par Recall décroissant)
      Modèle  TP  FN   FP   TN % pannes détectées % pannes manquées  Recall  Precision  F1-score  AUC-ROC  Accuracy
    LR tuned 336   0 9674    0             100.0%              0.0%  1.0000     0.0336    0.0650   0.4887    0.0336
    RF tuned 336   0 9674    0             100.0%              0.0%  1.0000     0.0336    0.0650   0.4902    0.0336
   XGB tuned 286  50 7656 2018              85.1%             14.9%  0.8512     0.0360    0.0691   0.5046    0.2302
 LR baseline 232 104 6703 2971              69.0%             31.0%  0.6905     0.0335    0.0638   0.4931    0.3200
 RF baseline  46 290 2074 7600              13.7%             86.3%  0.1369     0.0217    0.0375   0.4610    0.7638
XGB baseline  29 307  748 8926               8.6%             91.4%  0.0863     0.0373    0.0521   0.5017    0.8946

--- Meilleur par critère ---
  Recall              : LR tuned  (1.0)
  F1-score            : XGB tuned  (0.0691)
  Precis

In [146]:
with open("C:/indusense/Sprint2/resultats_modeles.txt", "w", encoding="utf-8") as f:
    f.write(synthese.to_string(index=False))